# 00 - Códigos de liga por temporada, a partir de un club
Entrada: ID de clubes que cubran el rango deseado tanto en primera como en segunda división (se ven en la URL de la página del club).

Cadena: calendario del club en cada temporada -> códigos de competición de ese año -> chequeo en la página de liga (¿está el club?) -> tabla de períodos por código.

Escribe `ligas_spielplan_semilla` y `ligas_codigos_por_anio` en `workspace.inferiores`. No toca ninguna otra tabla.

## Contenido

| Sección | Qué hace | Escribe |
|---|---|---|
| 1 | Funciones de descarga con caché y parámetros (`SEMILLAS`, `ANIOS`) | nada |
| 2 | Baja el calendario de cada club semilla en cada temporada y extrae los códigos de competición | `ligas_spielplan_semilla` |
| 3 | Resume en qué años aparece cada código | nada |
| 4 | Descarta los códigos cuya página de liga no incluye al club | nada |
| 5 | Chequea año por año las páginas de liga de los códigos conservados | `ligas_codigos_por_anio` |
| 6 | Consultas con el resultado: períodos y títulos por código | nada |

## Reglas de acceso al sitio que aplica el código

- Descarga `robots.txt` de `www.transfermarkt.com` al inicio y, antes de cada pedido no cacheado, consulta si la ruta está permitida. Si no lo está, lanza `RuntimeError` y no hace el pedido.
- Espera 3 s (`DELAY`) antes de cada pedido no cacheado.
- Ante HTTP `202`, `403` o `429` lanza `RuntimeError` y corta el notebook. No intenta resolver ningún desafío.
- Guarda cada página descargada en un Volume de Databricks. Una segunda corrida lee del Volume y no vuelve a pedir esas páginas.

## 1. Funciones y parámetros

**Constantes**

| Nombre | Valor |
|---|---|
| `TM` | `https://www.transfermarkt.com` |
| `UA` | User-Agent de navegador que se envía en cada pedido |
| `CACHE` | `/Volumes/workspace/inferiores/scraper/cache` |
| `DELAY` | `3.0` segundos entre pedidos |

**Funciones**

| Función | Qué hace |
|---|---|
| `curl(url, out)` | Descarga `url` con `curl` (máximo 40 s) al archivo `out` y devuelve el código HTTP como texto |
| `get(url)` | Devuelve `(código, html)`. Si la página está en caché, la lee del Volume. Si no, revisa `robots.txt`, espera `DELAY` y descarga con hasta 3 intentos. `200`: la guarda en caché. `202`/`403`/`429`: lanza `RuntimeError`. `404`: devuelve `("404", "")`. Otro código: espera 5 s, 10 s y 15 s entre intentos |
| `titulo(html)` | Texto de la etiqueta `<title>` |
| `etiqueta(title)` | Etiqueta de temporada (`NN/NN` o `NNNN`) que aparece antes del primer `\|` del título |
| `pagina_liga(code, y)` | Pide la página de la competición `code` en la temporada `y` y devuelve `(código HTTP, título, pares (nombre, id) de los clubes que aparecen)` |

`CLUB_RE` es la expresión regular que extrae los pares (nombre, id) de los enlaces de club.

**Parámetros que se editan acá**

| Parámetro | Contenido |
|---|---|
| `SEMILLAS` | IDs de club tal como figuran en la URL de su página |
| `ANIOS` | Temporadas a recorrer, de 2026 a 1985 en orden descendente |

In [ ]:
import subprocess, re, time, hashlib, os, urllib.robotparser
from collections import defaultdict
from pyspark.sql import Row

TM = "https://www.transfermarkt.com"
UA = "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120 Safari/537.36"
CACHE = "/Volumes/workspace/inferiores/scraper/cache"
DELAY = 3.0

def curl(url, out):
    r = subprocess.run(["curl", "-sL", "--max-time", "40", "-A", UA, "-w", "%{http_code}", "-o", out, url],
                       capture_output=True, text=True, timeout=60)
    return r.stdout.strip()

curl(TM + "/robots.txt", "/tmp/robots.txt")
rp = urllib.robotparser.RobotFileParser()
rp.parse(open("/tmp/robots.txt", encoding="utf-8", errors="replace").read().splitlines())

def get(url):
    p = f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}.html"
    if os.path.exists(p):
        return "200", open(p, encoding="utf-8", errors="replace").read()
    if not rp.can_fetch(UA, url):
        raise RuntimeError(f"robots.txt prohibe: {url}")
    time.sleep(DELAY)
    tmp = p + ".tmp"
    code = ""
    for attempt in range(3):
        code = curl(url, tmp)
        if code == "200":
            os.replace(tmp, p)
            return code, open(p, encoding="utf-8", errors="replace").read()
        if code in ("202", "403", "429"):
            raise RuntimeError(f"HTTP {code} en {url}: freno")
        if code == "404":
            return code, ""
        time.sleep(5 * (attempt + 1))
    return code or "0", ""

CLUB_RE = re.compile(r'href="/([^/"]+)/startseite/verein/(\d+)')

def titulo(html):
    t = re.search(r"<title>([^<]*)</title>", html)
    return t.group(1).strip() if t else ""

def etiqueta(title):
    m = re.search(r"(\d{2}/\d{2}|\d{4})\s*\|", title)
    return m.group(1) if m else ""

def pagina_liga(code, y):
    c, html = get(f"{TM}/-/startseite/wettbewerb/{code}/plus/?saison_id={y}")
    return c, titulo(html), sorted(set(CLUB_RE.findall(html)))

# Parámetros: IDs de club (de la URL de su página) y rango de temporadas a recorrer.
SEMILLAS = ["189", "4557", "10534", "12362"]
ANIOS = list(range(2026, 1984, -1))

## 2. Calendario del club en cada temporada
Una fila por (club, `saison_id`). `repetida` = mismo título y mismos códigos que un año ya visto (página de respaldo).

Para cada club semilla y cada año de `ANIOS`, baja la página `spielplan` (calendario) y extrae los códigos de competición que aparecen en sus enlaces.

Escribe `workspace.inferiores.ligas_spielplan_semilla` (reemplaza la tabla completa) con las columnas `semilla`, `saison_id`, `http`, `title` (primeros 80 caracteres), `season_label`, `codigos` (separados por coma) y `repetida`.

In [ ]:
sp = []
for sid in SEMILLAS:
    vistos = set()
    for y in ANIOS:
        c, html = get(f"{TM}/-/spielplan/verein/{sid}/saison_id/{y}")
        t = titulo(html)
        codigos = sorted(set(re.findall(r'/wettbewerb/([A-Z0-9]+)', html)))
        clave = (t, tuple(codigos))
        repetida = clave in vistos
        vistos.add(clave)
        sp.append(Row(semilla=sid, saison_id=y, http=int(c) if c.isdigit() else 0,
                      title=t[:80], season_label=etiqueta(t), codigos=",".join(codigos), repetida=repetida))
        print(sid, y, c, "REPETIDA" if repetida else "", etiqueta(t), codigos, flush=True)

spark.createDataFrame(sp).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.ligas_spielplan_semilla")

## 3. Qué años aparece cada código
Arma, para cada par (club semilla, código), el conjunto de años en que aparece. Ignora las filas `repetida` y las que no devolvieron HTTP 200. Imprime cuántos años tiene cada par y su año mínimo y máximo. No escribe tablas.

In [ ]:
anios_por_codigo = defaultdict(set)
for r in sp:
    if r.repetida or r.http != 200:
        continue
    for k in r.codigos.split(","):
        if k:
            anios_por_codigo[(r.semilla, k)].add(r.saison_id)

for (sid, k), ys in sorted(anios_por_codigo.items()):
    print(sid, k, "| años:", len(ys), "| de", min(ys), "a", max(ys))

## 4. Filtro: ¿la página de liga de ese código incluye al club?
Prueba hasta 3 años por código (el menor, el del medio y el mayor). Los códigos de navegación (otras ligas, copas) no incluyen al club y se descartan.

El código se conserva en cuanto uno de los años probados incluye al club. Si ninguno lo incluye, se descarta. Al final imprime cuántas páginas de liga de los códigos conservados todavía no están en caché, con una estimación de 3 s por página. No escribe tablas.

In [ ]:
def tres(ys):
    ys = sorted(ys)
    return sorted({ys[0], ys[len(ys) // 2], ys[-1]})

conservados = {}
for (sid, k), ys in sorted(anios_por_codigo.items()):
    ok = False
    for y in tres(ys):
        c, t, found = pagina_liga(k, y)
        if sid in {cid for _, cid in found}:
            ok = True
            break
    print(sid, k, "CONSERVADO" if ok else "descartado", flush=True)
    if ok:
        conservados[(sid, k)] = ys

paginas_nuevas = 0
for (sid, k), ys in conservados.items():
    for y in ys:
        url = f"{TM}/-/startseite/wettbewerb/{k}/plus/?saison_id={y}"
        if not os.path.exists(f"{CACHE}/{hashlib.md5(url.encode()).hexdigest()}.html"):
            paginas_nuevas += 1
print("páginas de liga a pedir en la celda 5 (no cacheadas):", paginas_nuevas, "≈", paginas_nuevas * 3, "s")

## 5. Chequeo año por año
`club_en_liga` = el club figura en la página de ese código y año. `titulo_repetido` = el título ya apareció en un año mayor del mismo código (página de respaldo).

Recorre todos los años de cada código conservado, del más nuevo al más viejo, y escribe `workspace.inferiores.ligas_codigos_por_anio` (reemplaza la tabla completa) con las columnas `semilla`, `code`, `saison_id`, `http`, `title` (primeros 80 caracteres), `season_label`, `n_clubs`, `club_en_liga` y `titulo_repetido`.

In [ ]:
res = []
for (sid, k), ys in sorted(conservados.items()):
    vistos_t = set()
    for y in sorted(ys, reverse=True):
        c, t, found = pagina_liga(k, y)
        rep = t in vistos_t
        vistos_t.add(t)
        res.append(Row(semilla=sid, code=k, saison_id=y, http=int(c) if c.isdigit() else 0,
                       title=t[:80], season_label=etiqueta(t), n_clubs=len(found),
                       club_en_liga=sid in {cid for _, cid in found}, titulo_repetido=rep))

spark.createDataFrame(res).write.mode("overwrite").option("overwriteSchema", "true") \
    .saveAsTable("workspace.inferiores.ligas_codigos_por_anio")

## 6. Resultado: períodos por código
Una fila por (club semilla, código): primer y último `saison_id`, cantidad de años, etiquetas de temporada del primer y último año, y lista de años. Cuenta solo las filas con `club_en_liga`, sin `titulo_repetido` y con HTTP 200.

In [ ]:
display(spark.sql("""
  SELECT semilla, code,
         MIN(saison_id) AS saison_desde, MAX(saison_id) AS saison_hasta, COUNT(*) AS anios,
         MIN_BY(season_label, saison_id) AS etiqueta_desde, MAX_BY(season_label, saison_id) AS etiqueta_hasta,
         ARRAY_SORT(COLLECT_LIST(saison_id)) AS saisons
  FROM workspace.inferiores.ligas_codigos_por_anio
  WHERE club_en_liga AND NOT titulo_repetido AND http = 200
  GROUP BY semilla, code
  ORDER BY semilla, saison_desde"""))

### Títulos por código
Una fila por código: años distintos, título de la temporada más vieja y de la más nueva, y máximo de clubes que figuraron en la página. Mismos filtros que la consulta anterior.

In [ ]:
display(spark.sql("""
  SELECT code,
         COUNT(DISTINCT saison_id) AS anios_distintos,
         MIN_BY(title, saison_id) AS titulo_mas_viejo,
         MAX_BY(title, saison_id) AS titulo_mas_nuevo,
         MAX(n_clubs) AS max_clubes
  FROM workspace.inferiores.ligas_codigos_por_anio
  WHERE club_en_liga AND NOT titulo_repetido AND http = 200
  GROUP BY code ORDER BY code"""))